*0.3 Classical NLP*

# TF-IDF

**The situation.** Keyword search returns the longest articles first, because they contain every query word somewhere. And a query for "refund" matches every article that says "the" — the search counts matches without asking how *telling* each word is.

**TF-IDF.** Score each word in each document by two things: how often it appears in that document (*term frequency*) and how rare it is across all documents (*inverse document frequency*). "the" appears everywhere → weight near 0. "refund" appears in 3 of 4,000 articles → high weight where it appears. Documents become vectors of these weights; search is cosine similarity between the query vector and each document vector.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer

articles = [
    "How to request a refund for a duplicate charge",
    "Why was I charged twice for the same order",
    "Change the email address on your account",
    "The refund policy for the annual plan and the monthly plan and the team plan",
]
vectorizer = TfidfVectorizer()
document_vectors = vectorizer.fit_transform(articles)
words = vectorizer.get_feature_names_out()

print("vocabulary size:", len(words))
for word in ("the", "refund", "charge"):
    column = list(words).index(word)
    print(f"idf({word!r}) = {vectorizer.idf_[column]:.2f}")
assert vectorizer.idf_[list(words).index("the")] < vectorizer.idf_[list(words).index("charge")]

vocabulary size: 26
idf('the') = 1.22
idf('refund') = 1.51
idf('charge') = 1.92


**Reading the output.** "the" has the lowest idf — it appears in most articles, so it counts for little. "charge" appears in fewer, so it counts for more. The weights are learned from the corpus, not from a stopword list.

**Search with it.**

In [3]:
from sklearn.metrics.pairwise import cosine_similarity

query_vector = vectorizer.transform(["refund for duplicate charge"])
scores = cosine_similarity(query_vector, document_vectors).ravel()
ranked = sorted(zip(scores, articles), reverse=True)
for score, article in ranked:
    print(f"{score:.3f}  {article}")
assert ranked[0][1].startswith("How to request a refund")

0.709  How to request a refund for a duplicate charge
0.119  The refund policy for the annual plan and the monthly plan and the team plan
0.090  Why was I charged twice for the same order
0.000  Change the email address on your account


**Reading the output.** The short, on-topic article wins. The long policy article that repeats "the" and "plan" scores lower even though it also contains "refund" — cosine normalises for length and "the" weighs almost nothing.

```
weight(word, doc) = tf(word in doc) × idf(word)        idf = log(N / docs containing word)
"the"     tf high  × idf ≈ 0     → ≈ 0
"refund"  tf 1     × idf high    → high
```

**The rule to remember.** TF-IDF turns text into vectors where telling words weigh more. It is still the fastest, cheapest baseline for search and classification, and often hard to beat on exact-term queries.

| Use it when | Don't when | Instead use |
|---|---|---|
| keyword search, a classification baseline, feature extraction | the query and the document use different words for the same thing | embeddings (0.2) or hybrid search; BM25 (next) for search specifically |

**Watch out**
- `fit` on the corpus once and save the vectorizer; `transform` queries with the same one. Re-fitting changes every weight.
- Vocabulary is fixed at fit time; new words in queries are ignored silently.
- Sublinear tf (`sublinear_tf=True`) and n-grams are the two settings most worth trying.